In [1]:
#set dataframe
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104],
    "customer_name": ["Ali", "Sara", "Reza", "Mina"],
    "city": ["Tehran", "Shiraz", "Isfahan", "Tabriz"]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": [101, 102, 101, 105],
    "amount": [500, 750, 300, 900]
})

display(customers)
display(orders)

,customer_id,customer_name,city
0,101,Ali,Tehran
1,102,Sara,Shiraz
2,103,Reza,Isfahan
3,104,Mina,Tabriz


,order_id,customer_id,amount
0,1001,101,500
1,1002,102,750
2,1003,101,300
3,1004,105,900


In [2]:
#Review the four join types
inner = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='inner'
)
display(inner)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001,500
1,101,Ali,Tehran,1003,300
2,102,Sara,Shiraz,1002,750


In [3]:
left = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='left'
)
display(left)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN


In [4]:
right = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='right'
)
display(right)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001,500
1,102,Sara,Shiraz,1002,750
2,101,Ali,Tehran,1003,300
3,105,NaN,NaN,1004,900


In [5]:
outer = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='outer'
)
display(outer)

,customer_id,customer_name,city,order_id,amount
0,101,Ali,Tehran,1001.0,500.0
1,101,Ali,Tehran,1003.0,300.0
2,102,Sara,Shiraz,1002.0,750.0
3,103,Reza,Isfahan,NaN,NaN
4,104,Mina,Tabriz,NaN,NaN
5,105,NaN,NaN,1004.0,900.0


| Join | What it preserves | Typical use |
|---|---|---|
| `inner` | Matching keys only | Analyse matched transactions |
| `left` | All left-side rows | Analyse all customers |
| `right` | All right-side rows | Preserve all orders |
| `outer` | All rows from both sides | Audit datasets |

In [7]:
#Merge using multiple keys
targets = pd.DataFrame({
    "product": ["Laptop", "Laptop", "Phone", "Phone"],
    "region": ["East", "West", "East", "West"],
    "target": [100, 150, 200, 250]
})

display(targets)

,product,region,target
0,Laptop,East,100
1,Laptop,West,150
2,Phone,East,200
3,Phone,West,250


In [8]:
sales = pd.DataFrame({
    "product": ["Laptop", "Laptop", "Phone", "Phone"],
    "region": ["East", "West", "East", "West"],
    "actual_sales": [90, 170, 220, 230]
})

display(sales)

,product,region,actual_sales
0,Laptop,East,90
1,Laptop,West,170
2,Phone,East,220
3,Phone,West,230


In [9]:
#We want to compare actual sales with their corresponding targets.
result = pd.merge(
    sales,
    targets,
    on=['product','region'],
    how='inner'
)
display(result)

,product,region,actual_sales,target
0,Laptop,East,90,100
1,Laptop,West,170,150
2,Phone,East,220,200
3,Phone,West,230,250


Pandas matches a row only when both columns match.

In [10]:
#Calculate the difference
result['difference'] = result['actual_sales'] - result['target']
display(result)

,product,region,actual_sales,target,difference
0,Laptop,East,90,100,-10
1,Laptop,West,170,150,20
2,Phone,East,220,200,20
3,Phone,West,230,250,-20


In [11]:
#Now identify products and regions that missed their targets:
missed = result.loc[result['difference'] < 0]
display(missed)

,product,region,actual_sales,target,difference
0,Laptop,East,90,100,-10
3,Phone,West,230,250,-20


In [13]:
#One-to-many relationships and duplicate keys
customers = pd.DataFrame({
    "customer_id": [101, 102],
    "customer_name": ["Ali", "Sara"]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": [101, 101, 101, 102],
    "amount": [100, 200, 300, 400]
})
display(customers)
display(orders)

,customer_id,customer_name
0,101,Ali
1,102,Sara


,order_id,customer_id,amount
0,1001,101,100
1,1002,101,200
2,1003,101,300
3,1004,102,400


In [14]:
merged = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='left'
)
display(merged)

,customer_id,customer_name,order_id,amount
0,101,Ali,1001,100
1,101,Ali,1002,200
2,101,Ali,1003,300
3,102,Sara,1004,400


This is one-to-many merging: one customer can have many orders.

In [15]:
#What if both DataFrames contain repeated keys?
left_df = pd.DataFrame({
    "key": ["A", "A"],
    "left_value": [10, 20]
})

right_df = pd.DataFrame({
    "key": ["A", "A", "A"],
    "right_value": [100, 200, 300]
})

merged = pd.merge(
    left_df,
    right_df,
    on='key',
    how='inner'
)
display(merged)

,key,left_value,right_value
0,A,10,100
1,A,10,200
2,A,10,300
3,A,20,100
4,A,20,200
5,A,20,300


This is called a many-to-many merge.
It isn't necessarily wrong, but it can inflate totals if you don't understand the relationship.

A practical warning
Imagine the left table contains a sales total of 1,000, and the right table contains three matching records. A many-to-many merge might repeat the 1,000 across multiple rows.
Before merging, ask:
1. Is the key unique in the left DataFrame?
2. Is the key unique in the right DataFrame?
3. Should the relationship be one-to-one, one-to-many, or many-to-many?

In [16]:
#Validate merges with validate
#It checks whether the key relationships match your expectations.
pd.merge(
    left_df,
    right_df,
    on='key',
    validate='one_to_one'
)

MergeError: Merge keys are not unique in either left or right dataset; not a one-to-one merge

| validate value | Left key | Right key |
|---|---|---|
| `"one_to_one"` | Unique | Unique |
| `"one_to_many"` | Unique | May repeat |
| `"many_to_one"` | May repeat | Unique |
| `"many_to_many"` | May repeat | May repeat |

In [20]:
#Use indicator=True to investigate unmatched rows
customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104],
    "customer_name": ["Ali", "Sara", "Reza", "Mina"],
    "city": ["Tehran", "Shiraz", "Isfahan", "Tabriz"]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": [101, 102, 101, 105],
    "amount": [500, 750, 300, 900]
})

audit = pd.merge(
    customers,
    orders,
    on='customer_id',
    how='outer',
    indicator=True
)
display(audit)

,customer_id,customer_name,city,order_id,amount,_merge
0,101,Ali,Tehran,1001.0,500.0,both
1,101,Ali,Tehran,1003.0,300.0,both
2,102,Sara,Shiraz,1002.0,750.0,both
3,103,Reza,Isfahan,NaN,NaN,left_only
4,104,Mina,Tabriz,NaN,NaN,left_only
5,105,NaN,NaN,1004.0,900.0,right_only


Pandas adds a _merge column with one of three values:
| `_merge` value | Meaning |
|---|---|
| `left_only` | The key exists only in the left DataFrame |
| `right_only` | The key exists only in the right DataFrame |
| `both` | The key exists in both DataFrames |

In [21]:
#Find customers without orders
audit.loc[audit['_merge']=='left_only']

,customer_id,customer_name,city,order_id,amount,_merge
3,103,Reza,Isfahan,NaN,NaN,left_only
4,104,Mina,Tabriz,NaN,NaN,left_only


In [22]:
#Find orders without a matching customer
audit.loc[audit['_merge']=='right_only']

,customer_id,customer_name,city,order_id,amount,_merge
5,105,NaN,NaN,1004.0,900.0,right_only


In [23]:
#Count the match types
audit['_merge'].value_counts()

_merge
both          3
left_only     2
right_only    1
Name: count, dtype: int64

In [24]:
#Mini project: Sales target analysis
targets = pd.DataFrame({
    "region": ["North", "South", "East", "West"],
    "target": [1000, 1500, 1200, 1800]
})

actual_sales = pd.DataFrame({
    "region": ["North", "South", "East", "Central"],
    "actual_sales": [1100, 1300, 1400, 900]
})

display(targets)
display(actual_sales)

,region,target
0,North,1000
1,South,1500
2,East,1200
3,West,1800


,region,actual_sales
0,North,1100
1,South,1300
2,East,1400
3,Central,900


In [26]:
#Use an outer join so that no region disappears.
merged = pd.merge(
    targets,
    actual_sales,
    on='region',
    how='outer',
    validate='one_to_one',
    indicator=True
)
display(merged)

,region,target,actual_sales,_merge
0,Central,NaN,900.0,right_only
1,East,1200.0,1400.0,both
2,North,1000.0,1100.0,both
3,South,1500.0,1300.0,both
4,West,1800.0,NaN,left_only


In [27]:
#Investigate unmatched regions
merged.loc[merged['_merge'] != 'both']

,region,target,actual_sales,_merge
0,Central,NaN,900.0,right_only
4,West,1800.0,NaN,left_only


In [28]:
#Calculate sales performance
merged['difference'] = merged['actual_sales'] - merged['target']
display(merged)

,region,target,actual_sales,_merge,difference
0,Central,NaN,900.0,right_only,NaN
1,East,1200.0,1400.0,both,200.0
2,North,1000.0,1100.0,both,100.0
3,South,1500.0,1300.0,both,-200.0
4,West,1800.0,NaN,left_only,NaN


In [29]:
merged['achiment_percent'] = (
    merged['actual_sales'] /
    merged['target'] *
    100
).round(2)
display(merged)

,region,target,actual_sales,_merge,difference,achiment_percent
0,Central,NaN,900.0,right_only,NaN,NaN
1,East,1200.0,1400.0,both,200.0,116.67
2,North,1000.0,1100.0,both,100.0,110.00
3,South,1500.0,1300.0,both,-200.0,86.67
4,West,1800.0,NaN,left_only,NaN,NaN
